In [ ]:
%pip install -q deep-translator bert-score sentence-transformers nltk spacy

In [19]:
import time
import importlib
from deep_translator import MyMemoryTranslator
import nltk
import spacy
from nltk.corpus import wordnet as wn
from bert_score import score
import pandas as pd
from sentence_transformers import SentenceTransformer, util




nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("omw-2.0")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\igort\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\igort\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package omw-2.0 to
[nltk_data]     C:\Users\igort\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-2.0 is already up-to-date!


True

In [ ]:


def traduzir_frase(frase_original):
    frase = MyMemoryTranslator(source='portuguese', target='japanese').translate(frase_original)
    time.sleep(1)
    frase_volta_portugues = MyMemoryTranslator(source='japanese', target='portuguese').translate(frase)
    time.sleep(1)
    return frase_volta_portugues

def read_frases_from_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]



In [ ]:

nlp = spacy.load("pt_core_news_sm")

def contradizer(frase):
    doc = nlp(frase)
   
    if doc and doc[0].pos_ == "PROPN":
        doc = nlp(frase[:1].lower() + frase[1:])

    raiz = None
    for palavra in doc:
        if palavra.dep_ == "ROOT":
            raiz = palavra

    if raiz is None:
        return None

    for palavra in raiz.children:
        if palavra.lower_ == "não":
            nova = ""
            for p in doc:
                if p != palavra:
                    nova += p.text_with_ws
            return nova[0].upper() + nova[1:]


    alvo = None
    for palavra in raiz.children:
        if palavra.dep_ in ["aux", "aux:pass", "cop"] and palavra.i < raiz.i:
            alvo = palavra
            break

    if alvo is None:
        if raiz.pos_ == "VERB" or raiz.pos_ == "AUX":
            alvo = raiz
        else:
            return None

    if alvo.i == 0:
        return "Não " + frase[0].lower() + frase[1:]
    return frase[:alvo.idx] + "não " + frase[alvo.idx:]


In [32]:
def bert_score(originais, parafraseadas, contradicadas):

    P_par, R_par, F1_par = score(parafraseadas, originais, lang="pt", rescale_with_baseline=True)
    P_con, R_con, F1_con = score(contradicadas, originais, lang="pt", rescale_with_baseline=True)

    print(f"Paráfrases   -> P: {P_par.mean():.4f}  R: {R_par.mean():.4f}  F1: {F1_par.mean():.4f}")
    print(f"Contradições -> P: {P_con.mean():.4f}  R: {R_con.mean():.4f}  F1: {F1_con.mean():.4f}")

    resultados = pd.DataFrame({
        'original': originais,
        'parafraseada': parafraseadas,
        'F1_parafrase': F1_par.numpy(),
        'contradicao': contradicadas,
        'F1_contradicao': F1_con.numpy(),
    })
    return resultados

def sentence_transformers_score(originais, parafraseadas, contradicadas):

    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    emb_ori = model.encode(originais, convert_to_tensor=True)
    emb_par = model.encode(parafraseadas, convert_to_tensor=True)
    emb_con = model.encode(contradicadas, convert_to_tensor=True)

    sim_par = util.pairwise_cos_sim(emb_ori, emb_par).cpu()
    sim_con = util.pairwise_cos_sim(emb_ori, emb_con).cpu()

    print(f"Paráfrases   -> Cosseno: {sim_par.mean():.4f}")
    print(f"Contradições -> Cosseno: {sim_con.mean():.4f}")

    resultados = pd.DataFrame({
        'original': originais,
        'parafraseada': parafraseadas,
        'sim_parafrase': sim_par.numpy(),
        'contradicao': contradicadas,
        'sim_contradicao': sim_con.numpy(),
    })
    return resultados

In [33]:
def main():
    frases = read_frases_from_file('frases.txt')

    try:
        frases_parafraseada = read_frases_from_file('frases_parafraseadas.txt')
        frases_contradicadas = read_frases_from_file('frases_contradicadas.txt')
    except FileNotFoundError:
        frases_parafraseada = []
        frases_contradicadas = []

    if frases_parafraseada and frases_contradicadas:
        resultados_bert_score = bert_score(frases, frases_parafraseada, frases_contradicadas)
        resultados_sbert = sentence_transformers_score(frases, frases_parafraseada, frases_contradicadas)
    else:    
        frases_parafraseada = []
        frases_contradicadas = []

        for frase in frases:
            frases_contradicadas.append(contradizer(frase) or '')
            frases_parafraseada.append(traduzir_frase(frase))
            


        with open('frases_parafraseadas.txt', 'w', encoding='utf-8') as f:
            for frase in frases_parafraseada:
                f.write(frase + '\n')

        with open('frases_contradicadas.txt', 'w', encoding='utf-8') as f:
            for frase in frases_contradicadas:
                f.write((frase or '') + '\n')

        resultados_bert_score = bert_score(frases, frases_parafraseada, frases_contradicadas)
        resultados_sbert = sentence_transformers_score(frases, frases_parafraseada, frases_contradicadas)
    
main()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 884.09it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 708.80it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                      

Paráfrases   -> P: 0.7519  R: 0.7366  F1: 0.7442
Contradições -> P: 0.9198  R: 0.9530  F1: 0.9364


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2955.63it/s]


Paráfrases   -> Cosseno: 0.9319
Contradições -> Cosseno: 0.8111
